# 47 — M5 Gate

Formal closure gate for Advanced Data Science & Robustness. It validates evidence, lineage, fresh OOT governance, documentation and the explicit non-authorization of production.


In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

from argos.evaluation.m5_gate import evaluate_m5_gate, persist_m5_gate


In [ ]:
report, decision = evaluate_m5_gate(
    root=paths.root,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
    schema_version=M5_SPEC.schema_version,
)
report


In [ ]:
report_path, decision_path = persist_m5_gate(
    report=report,
    decision=decision,
    reports_root=paths.reports,
    decision_path=paths.docs / "decisions" / "M5_GATE.md",
)
print(f"M5 {'✅ CLOSED' if decision.closed else '❌ OPEN'}")
print(f"Blocking checks: {decision.blocking_passed}/{decision.blocking_total}")
print("Gate report:", report_path)
print("Decision artifact:", decision_path)
assert decision.closed, f"M5 permanece OPEN: {decision.failed_checks}"
